# NYC Taxi Trip Duration — Exploratory Data Analysis

**Data Source:** NYC TLC (Taxi & Limousine Commission) — [nyc.gov/tlc](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

This notebook explores the raw NYC TLC yellow taxi data before model training.

## Contents
1. Load & Preview Data
2. Missing Values & Data Quality
3. Trip Duration Distribution
4. Distance vs Duration Analysis
5. Time-of-Day Patterns
6. Borough-level Analysis
7. Feature Correlation
8. Key Insights for Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('dark_background')
YELLOW = '#F7C900'
sns.set_palette([YELLOW, '#FF4444', '#00D4FF', '#00FF88'])

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
# Load a sample (first 200k rows for exploration speed)
df = pd.read_parquet('../data/yellow_tripdata_2024-01.parquet')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Data types and nulls
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

## 2. Trip Duration Distribution

In [ ]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

# Filter valid trips
valid = df[(df['duration_min'] >= 1) & (df['duration_min'] <= 120)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

valid['duration_min'].hist(bins=80, ax=axes[0], color=YELLOW, edgecolor='none', alpha=0.9)
axes[0].set_title('Trip Duration Distribution', color=YELLOW, fontsize=14)
axes[0].set_xlabel('Minutes')
axes[0].axvline(valid['duration_min'].median(), color='red', linestyle='--', label=f'Median: {valid["duration_min"].median():.1f} min')
axes[0].legend()

# Log scale
np.log1p(valid['duration_min']).hist(bins=80, ax=axes[1], color='#00D4FF', edgecolor='none', alpha=0.9)
axes[1].set_title('Duration (Log Scale)', color=YELLOW, fontsize=14)
axes[1].set_xlabel('log(1 + minutes)')

plt.tight_layout()
plt.savefig('../notebooks/duration_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(valid['duration_min'].describe().round(2))

## 3. Hourly Traffic Patterns

In [ ]:
valid['hour'] = valid['tpep_pickup_datetime'].dt.hour
valid['day_of_week'] = valid['tpep_pickup_datetime'].dt.dayofweek

# Average duration by hour
hourly = valid.groupby('hour')['duration_min'].agg(['mean', 'median', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hourly.index, hourly['mean'], color=YELLOW, alpha=0.85)
axes[0].set_title('Avg Trip Duration by Hour of Day', color=YELLOW, fontsize=13)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg Duration (min)')
axes[0].axvspan(7, 9, alpha=0.15, color='red', label='Morning Rush')
axes[0].axvspan(16, 19, alpha=0.15, color='orange', label='PM Rush')
axes[0].legend()

# Trip volume by hour
axes[1].bar(hourly.index, hourly['count'] / 1000, color='#00D4FF', alpha=0.85)
axes[1].set_title('Trip Volume by Hour (thousands)', color=YELLOW, fontsize=13)
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Trips (K)')

plt.tight_layout()
plt.savefig('../notebooks/hourly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distance vs Duration Correlation

In [ ]:
sample = valid.sample(min(5000, len(valid)), random_state=42)

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    sample['trip_distance'],
    sample['duration_min'],
    c=sample['hour'],
    cmap='plasma',
    alpha=0.4,
    s=12,
)
plt.colorbar(sc, label='Hour of Day')
plt.xlim(0, 30)
plt.ylim(0, 90)
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Trip Duration (minutes)')
plt.title('Distance vs Duration (colored by hour)', color=YELLOW, fontsize=13)
plt.savefig('../notebooks/distance_duration.png', dpi=150, bbox_inches='tight')
plt.show()

corr = valid[['trip_distance', 'duration_min', 'hour', 'fare_amount']].corr()
print('\nCorrelation with duration_min:')
print(corr['duration_min'].sort_values(ascending=False))

## 5. Key Insights

In [ ]:
print('=== KEY INSIGHTS FOR FEATURE ENGINEERING ===')
print()
print(f'1. Median trip duration: {valid["duration_min"].median():.1f} minutes')
print(f'2. Median trip distance: {valid["trip_distance"].median():.1f} miles')

rush = valid[valid['hour'].isin([7,8,9,16,17,18])]['duration_min'].mean()
off_peak = valid[~valid['hour'].isin([7,8,9,16,17,18])]['duration_min'].mean()
print(f'3. Rush hour avg: {rush:.1f} min vs off-peak: {off_peak:.1f} min ({rush/off_peak:.1f}x longer)')

weekday = valid[valid['day_of_week'] < 5]['duration_min'].mean()
weekend = valid[valid['day_of_week'] >= 5]['duration_min'].mean()
print(f'4. Weekday avg: {weekday:.1f} min vs Weekend: {weekend:.1f} min')
print()
print('Recommended features: distance, hour (cyclical), rush_hour flag, weekend flag,')
print('  airport_trip flag, borough zones, passenger_count')